# C2-linear-models — Practice p24

**Type:** challenge · **Difficulty:** advanced · **Concepts:** ols-rank-identifiability-and-pseudoinverse

\[
X=\begin{pmatrix}
1&0&1\\1&1&2\\1&2&3\\1&3&4\\1&4&5
\end{pmatrix},\qquad
y=\begin{pmatrix}1\\2\\2\\4\\5\end{pmatrix}.
\]

**(a)** State the exact column relation, find a nonzero integer
$z\in\mathcal N(X)$, and prove rank is $2$. Store \`z_p24\`.

**(b)** Compute $\beta^+=X^+y$. Prove every
$\beta^++tz$, $t\in\mathbb R$, has the same predictions/residual and
solves the normal equations. Prove conversely that every minimizer
differs from $\beta^+$ by a nullspace vector, so this one-dimensional
family is complete.

**(c)** Prove $\beta^+$ lies in the row space and is orthogonal to $z$.
Expand $\lVert\beta^++tz\rVert_2^2$ and prove its unique minimum is at
$t=0$. Sampling alone is not a proof.

**(d)** Without loops, build all supplied family members, predictions,
residuals, norms, and normal-equation gaps. Certify common predictions,
orthogonality, and minimum at zero with
\`ATOL = 1e-10\`, \`RTOL = 0.0\`.

*Your exact relation and proofs for (a)–(c) here.*

Let the columns be $c_1,c_2,c_3$. Row by row, $c_3=c_1+c_2$, so

$$
z=\begin{pmatrix}1\\1\\-1\end{pmatrix}
\in\mathcal N(X).
$$

The first two columns are independent because $c_1$ is constant while $c_2=(0,1,2,3,4)^T$ is not; together with the one column relation, this proves $\operatorname{rank}(X)=2$ and $\mathcal N(X)=\operatorname{span}\{z\}$.

The pseudoinverse coefficient is

$$
\beta^+=X^+y=\frac15\begin{pmatrix}1\\2\\3\end{pmatrix}.
$$

For every $t\in\mathbb R$, $X(\beta^++tz)=X\beta^+$ because $Xz=0$; therefore every family member has the same predictions and residual. Since $\beta^+$ is a least-squares solution, $X^T(X\beta^+-y)=0$, so the identical residual shows every family member solves the normal equations too.

Conversely, if $\tilde\beta$ is any minimizer, both $\tilde\beta$ and $\beta^+$ satisfy the normal equations. Subtracting gives

$$
X^TX(\tilde\beta-\beta^+)=0.
$$

Multiplication on the left by $(\tilde\beta-\beta^+)^T$ yields $\lVert X(\tilde\beta-\beta^+)\rVert_2^2=0$, hence $\tilde\beta-\beta^+\in\mathcal N(X)=\operatorname{span}\{z\}$. Thus the displayed one-parameter family contains every minimizer.

From the SVD formula $X^+=V_r\Sigma_r^{-1}U_r^T$, $\beta^+$ lies in the span of the right singular vectors with nonzero singular values, which is the row space of $X$. The row space equals $\mathcal N(X)^\perp$, so $(\beta^+)^Tz=0$; directly, $(1+2-3)/5=0$. Therefore

$$
\begin{aligned}
\lVert\beta^++tz\rVert_2^2
&=\lVert\beta^+\rVert_2^2+2t(\beta^+)^Tz+t^2\lVert z\rVert_2^2\\
&=\frac{14}{25}+3t^2.
\end{aligned}
$$

This expression has the unique minimum at $t=0$, proving—not merely sampling—that $\beta^+$ is the unique minimum-norm member.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

X = np.array([[1.0, 0.0, 1.0],
              [1.0, 1.0, 2.0],
              [1.0, 2.0, 3.0],
              [1.0, 3.0, 4.0],
              [1.0, 4.0, 5.0]])
y = np.array([1.0, 2.0, 2.0, 4.0, 5.0])
t_grid = np.array([-3.0, -1.0, -0.25, 0.0, 0.5, 2.0, 4.0])

z_p24 = np.array([1, 1, -1], dtype=int)
beta_plus_p24 = np.linalg.pinv(X) @ y
beta_family_p24 = beta_plus_p24[None, :] + t_grid[:, None] * z_p24[None, :]
pred_family_p24 = beta_family_p24 @ X.T
resid_family_p24 = pred_family_p24 - y[None, :]
norms_p24 = np.linalg.norm(beta_family_p24, axis=1)
row_null_dot_p24 = float(beta_plus_p24 @ z_p24)
normal_gaps_p24 = np.max(np.abs(resid_family_p24 @ X), axis=1)

null_check_p24 = np.allclose(X @ z_p24, 0.0, atol=ATOL, rtol=RTOL)
pred_check_p24 = np.allclose(pred_family_p24, pred_family_p24[0], atol=ATOL, rtol=RTOL)
normal_check_p24 = np.allclose(normal_gaps_p24, 0.0, atol=ATOL, rtol=RTOL)
orthogonal_check_p24 = np.allclose(row_null_dot_p24, 0.0, atol=ATOL, rtol=RTOL)
minimum_index_p24 = int(np.argmin(norms_p24))

In [ ]:
assert np.array_equal(z_p24, np.array([1, 1, -1]))
assert np.linalg.matrix_rank(X) == 2
assert null_check_p24
assert np.allclose(beta_plus_p24, np.array([1 / 5, 2 / 5, 3 / 5]), atol=ATOL, rtol=RTOL)
assert beta_family_p24.shape == (len(t_grid), 3)
assert pred_family_p24.shape == (len(t_grid), 5)
assert resid_family_p24.shape == (len(t_grid), 5)
assert norms_p24.shape == (len(t_grid),)
assert normal_gaps_p24.shape == (len(t_grid),)
assert pred_check_p24 and normal_check_p24 and orthogonal_check_p24
assert np.allclose(norms_p24**2, 14 / 25 + 3 * t_grid**2, atol=ATOL, rtol=RTOL)
assert minimum_index_p24 == int(np.flatnonzero(t_grid == 0.0)[0])

### Answer check

The executable checks verify the exact null vector and rank, pseudoinverse coefficient $(1,2,3)^T/5$, vectorized family shapes, common predictions, normal-equation gaps, row/nullspace orthogonality, and the exact norm law $14/25+3t^2$ using `ATOL = 1e-10`, `RTOL = 0.0`. The written argument proves that the family is complete and that $t=0$ is uniquely minimum norm rather than inferring either fact from samples.